# BC5CDR Corpus Keyword Extraction with KeyLLM

Description: Extract BC5CDR Corpus abstract keywords using KeyLLM and calculate associated importance scores.

In [1]:
# Imports
import json
import openai
from keybert.llm import OpenAI
from keybert import KeyLLM
from sklearn.feature_extraction.text import CountVectorizer
import numpy as np
import pandas as pd

In [2]:
# Path to BC5CDR corpus abstracts
infile_path = '../1-preprocessing/bc5cdr-preprocessed-no-lex.json'

# Load the JSON contents into a Python variable
with open(infile_path, 'r') as file:
    docs = json.load(file)

n_docs = len(docs)
print(n_docs)

1500


In [3]:
# Extract keywords with KeyLLM

# Create LLM
prompt = """
I have the following biomedical article abstract:
[DOCUMENT]

Based on the information above, extract the biometical terminology terms that best describe the topic of the text.
Make sure to only extract terms that appear in the text.
Consider words connected by the underline charater as single words.
Return a comma-separated list of at most 25 single-word biomedical terms.
Use the following format separated by commas:
<term>
"""
client = openai.OpenAI(api_key="my_api_key") # Include API key here
llm = OpenAI(client, model="gpt-4.1-mini-2025-04-14", prompt=prompt, chat=True)

# Load it in KeyLLM
kw_model = KeyLLM(llm)

# Extract keywords
keywords_corpus = kw_model.extract_keywords(docs, check_vocab=True)

In [4]:
# Calculate KeyLLM keyword scores

# Flatten, extract, and alphebetize unique keywords
unique_raw_keywords = list({keyword for doc in keywords_corpus for keyword in doc})
unique_raw_keywords.sort()
n_unique_raw_keywords = len(unique_raw_keywords)

# Create master data frame
master_df = pd.DataFrame(data={
    'raw_keyword': unique_raw_keywords,
    'KeyLLM': np.zeros(n_unique_raw_keywords)
})

# Calculate keyword scores
for idx, raw_keyword in master_df['raw_keyword'].items():
    score = sum(1 for doc in keywords_corpus if raw_keyword in doc) / n_docs
    master_df.at[idx, 'KeyLLM'] = score

master_df['raw_keyword'] = master_df['raw_keyword'].str.lower()

display(master_df)

,raw_keyword,KeyLLM
0,,0.000667
1,112dihydro2acenaphthylenylpiperazine,0.000667
2,11deoxycortisol,0.000667
3,11dichloro222trifluoroethane,0.000667
4,11ketopregnenolone_sulphate,0.000667
...,...,...
8121,zonisamide,0.001333
8122,zopiclone,0.000667
8123,zuclopenthixol,0.000667
8124,zyban,0.000667


In [5]:
# Recover BC5CDR lexical units from raw keywords

# Read in BC5CDR data
bc5cdr_keyword_path = '../1-preprocessing/bc5cdr-keywords.tsv'
bc5cdr_df = pd.read_csv(bc5cdr_keyword_path, sep="\t")

# Convert lexical units to lowercase and strip '_lex' postfix
bc5cdr_df['raw_keyword'] = bc5cdr_df['lex'].str.lower().str.removesuffix('_lex')

# Add scores for lexical units
result_df = pd.merge(master_df, bc5cdr_df, on='raw_keyword', how='outer')

# Replace NaN scores with 0
result_df['KeyLLM'] = result_df['KeyLLM'].fillna(0)

# Clean up resulting data frame
result_df.rename(columns={'lex': 'term'}, inplace=True)

# Condition: If 'term' == NaN, take 'raw_keyword'; otherwise keep 'term'
# Includes non-lexical term scores.
result_df['term'] = np.where(result_df['term'].isna(), result_df['raw_keyword'], result_df['term'])

# Replace any NaNs in the 'sem' column with an empty string
# Uses empty string for the semantic class of non-lexical terms
result_df['sem'] = result_df['sem'].fillna(str())

# Drop 'raw_keywords' column
result_df.drop(columns=['raw_keyword'], inplace=True)

# Reorder columns
result_df = result_df.reindex(columns=['term', 'sem', 'KeyLLM'])

# Write to TSV
result_df.to_csv('keyllm-scores.tsv', sep='\t', index=False)

display(result_df)

,term,sem,KeyLLM
0,,,0.000667
1,112dihydro2acenaphthylenylpiperazine_lex,Chemical,0.000667
2,11deoxycortisol_lex,Chemical,0.000667
3,11dichloro222trifluoroethane_lex,Chemical,0.000667
4,11ketopregnenolone_sulphate_lex,Chemical,0.000667
...,...,...,...
9101,methicillin_lex,Chemical,0.000000
9102,tazobactam_lex,Chemical,0.000000
9103,galactose_lex,Chemical,0.000000
9104,dgalactose_lex,Chemical,0.000000
